# NexusBots — Function-Calling Benchmark

**Three systems compared on 100 holdout test rows:**
1. Heuristic Router (pure regex rules, 0ms, no GPU)
2. Qwen3-0.6B BASE (raw pretrained, zero-shot via system prompt)
3. Qwen3-0.6B FC (our fine-tuned model, 3 epochs QLoRA)

**Locked baseline (from `bench_function_calling.py` run):**

| System | Tool Acc | Arg F1 | UI Guide Acc |
|---|---|---|---|
| Heuristic Router | 0.45 | 0.17 | 0.27 |
| Qwen3-0.6B BASE | 0.61 | 0.46 | 0.26 |

In [2]:
import sys, os, json, re, time, csv
from pathlib import Path
from collections import defaultdict, Counter

FINETUNE_DIR = Path('/data1/cs24mtech14020/home/TDL/NexusBots-TDL-Project/finetune')
PROJECT_DIR  = FINETUNE_DIR.parent
sys.path.insert(0, str(FINETUNE_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'server' / 'ai'))

os.environ['CUDA_VISIBLE_DEVICES'] = '3'

from config import TEST_PATH, RESULTS_DIR, SYSTEM_PROMPT, ADAPTER_DIR, MERGED_DIR, MAX_SEQ_LENGTH
print(f'TEST_PATH    : {TEST_PATH}  exists={TEST_PATH.exists()}')
print(f'MERGED_DIR   : {MERGED_DIR}  exists={MERGED_DIR.exists()}')
print(f'ADAPTER_DIR  : {ADAPTER_DIR}  exists={ADAPTER_DIR.exists()}')

TEST_PATH    : /data1/cs24mtech14020/home/TDL/NexusBots-TDL-Project/finetune/data/test.jsonl  exists=True
MERGED_DIR   : /data1/cs24mtech14020/home/TDL/NexusBots-TDL-Project/finetune/output/qwen3-0_6b-fc-v1-merged  exists=True
ADAPTER_DIR  : /data1/cs24mtech14020/home/TDL/NexusBots-TDL-Project/finetune/output/qwen3-0_6b-fc-v1  exists=True


In [3]:
# ── Load test set ────────────────────────────────────────────────────────────
def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip(): rows.append(json.loads(line))
    return rows

test_rows = load_jsonl(TEST_PATH)
print(f'Loaded {len(test_rows)} test rows')
tools, langs = [], []
for row in test_rows:
    langs.append(row.get('language', 'en'))
    for msg in row.get('messages', []):
        if msg['role'] == 'assistant':
            tools.append(json.loads(msg['content'])['tool']); break
print('Tool distribution:', dict(Counter(tools)))
print('Lang distribution:', dict(Counter(langs)))

Loaded 100 test rows
Tool distribution: {'search_products': 25, 'get_product': 11, 'navigate_to': 27, 'recommend': 16, 'add_to_cart': 11, 'compare_products': 10}
Lang distribution: {'en': 55, 'hi': 27, 'te': 18}


In [4]:
# ── Scoring helpers ──────────────────────────────────────────────────────────
def extract_gold(row):
    for msg in row.get('messages', []):
        if msg['role'] == 'assistant':
            obj = json.loads(msg['content'])
            return obj['tool'], obj.get('arguments', {}), obj.get('ui_guide')
    return '', {}, None

def score_tool(p, g): return 1.0 if p == g else 0.0

def score_arg_f1(pred, gold):
    if not gold: return 1.0 if not pred else 0.0
    keys = set(gold) | set(pred)
    return sum(str(pred.get(k,'')).strip().lower() == str(gold.get(k,'')).strip().lower() for k in keys) / len(keys)

def score_ui(p, g): return 1.0 if p == g else 0.0

def parse_json(text):
    text = text.strip()
    try: return json.loads(text)
    except: pass
    s, e = text.find('{'), text.rfind('}')+1
    if s >= 0 and e > s:
        try: return json.loads(text[s:e])
        except: pass
    return {}

def evaluate(name, predict_fn, rows, ctx_is_str=True):
    by_lang = defaultdict(lambda: {'tc':0,'af':0.0,'ug':0,'n':0,'lat':[]})
    for row in rows:
        gold_tool, gold_args, gold_guide = extract_gold(row)
        lang = row.get('language','en')
        user_msg = next((m['content'] for m in row.get('messages',[]) if m['role']=='user'), '')
        parts = user_msg.split('\nContext: ', 1)
        query = parts[0].replace('Query: ','')
        ctx_str = parts[1] if len(parts)>1 else '{}'
        try:
            arg = ctx_str if ctx_is_str else (json.loads(ctx_str) if ctx_str else {})
            pred, lat = predict_fn(query, arg)
            pt, pa, pg = pred.get('tool',''), pred.get('arguments',{}), pred.get('ui_guide')
        except Exception as ex:
            print(f'  err row {row.get("id","?")}: {ex}')
            pt, pa, pg, lat = '', {}, None, 0.0
        b = by_lang[lang]
        b['tc'] += score_tool(pt, gold_tool); b['af'] += score_arg_f1(pa, gold_args)
        b['ug'] += score_ui(pg, gold_guide); b['n'] += 1; b['lat'].append(lat)
    summary = {}
    for lang, b in by_lang.items():
        n=b['n']; lat=sorted(b['lat'])
        summary[lang]={'tool_acc':round(b['tc']/n,4),'arg_f1':round(b['af']/n,4),'ui_guide_acc':round(b['ug']/n,4),'p50_ms':round(lat[n//2],1),'count':n}
    total=sum(b['n'] for b in by_lang.values())
    all_lat=sorted(l for b in by_lang.values() for l in b['lat'])
    summary['overall']={'tool_acc':round(sum(b['tc'] for b in by_lang.values())/total,4),'arg_f1':round(sum(b['af'] for b in by_lang.values())/total,4),'ui_guide_acc':round(sum(b['ug'] for b in by_lang.values())/total,4),'p50_ms':round(all_lat[len(all_lat)//2],1),'count':total}
    return summary

print('Scoring helpers ready.')

Scoring helpers ready.


In [5]:
# ── System 1: Heuristic Router ───────────────────────────────────────────────
# Pure keyword/regex rules. No model. No GPU. Instant.
# Fails on: multilingual, product-name→ID mapping, context-implicit queries.
def predict_heuristic(query, context):
    start = time.perf_counter(); q = query.lower()
    for pat,(t,a) in {
        r'\b(order|orders|my order|track|delivery)\b': ('navigate_to',{'page':'orders','params':{}}),
        r'\b(cart|basket|checkout)\b':                 ('navigate_to',{'page':'cart','params':{}}),
        r'\b(home|main page|start)\b':                 ('navigate_to',{'page':'home','params':{}}),
        r'\b(catalog|browse|all products|shop)\b':     ('navigate_to',{'page':'catalog','params':{}}),
    }.items():
        if re.search(pat,q): return {'tool':t,'arguments':a,'ui_guide':None},(time.perf_counter()-start)*1000
    if re.search(r'\b(add|put|cart)\b',q):
        ids=re.findall(r'\b([1-9]|1[0-2])\b',query); t,a='add_to_cart',{'product_id':int(ids[0]) if ids else 1}
    elif re.search(r'\b(compare|vs|versus|better)\b',q):
        ids=re.findall(r'\b([1-9]|1[0-2])\b',query); t,a='compare_products',{'product_id_1':int(ids[0]) if ids else 1,'product_id_2':int(ids[1]) if len(ids)>1 else 2,'focus':'specs'}
    elif re.search(r'\b(detail|info|about|what is|tell me about)\b',q):
        ids=re.findall(r'\b([1-9]|1[0-2])\b',query); t,a='get_product',{'product_id':int(ids[0]) if ids else 1}
    elif re.search(r'\b(recommend|suggest|best|budget|under|below)\b',q):
        bm=re.search(r'(\d{3,5})',query); budget=int(bm.group(1)) if bm else 5000
        cat=next((c for c in ['Kitchen','Home Cleaner','Drone','Humanoid'] if c.lower() in q),'')
        t,a='recommend',{'need':query,'budget':budget,'category':cat}
    else:
        cat=next((c for c in ['Kitchen','Home Cleaner','Drone','Humanoid'] if c.lower() in q),'')
        t,a='search_products',{'query':query,'category':cat}
    return {'tool':t,'arguments':a,'ui_guide':None},(time.perf_counter()-start)*1000

print('Running Heuristic Router...')
heuristic_results = evaluate('Heuristic Router', predict_heuristic, test_rows, ctx_is_str=False)
print('Overall:', heuristic_results['overall'])

Running Heuristic Router...
Overall: {'tool_acc': 0.45, 'arg_f1': 0.2522, 'ui_guide_acc': 0.27, 'p50_ms': 0.0, 'count': 100}


In [6]:
# ── System 2: Qwen3-0.6B BASE (zero-shot) ───────────────────────────────────
# Raw pretrained model, no task-specific training. Uses system prompt only.
import fc_model

fc_model._model = None; fc_model._tokenizer = None; fc_model._load_error = None
os.environ['FC_MODEL_PATH'] = '/data1/cs24mtech14020/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca'

loaded = fc_model.load_model()
print(f'BASE model loaded: {loaded}')

def predict_base(query, ctx_str):
    try: ctx = json.loads(ctx_str) if ctx_str else {}
    except: ctx = {}
    start = time.perf_counter()
    result = fc_model.predict_tool_call(query, 'en', ctx)
    lat = (time.perf_counter()-start)*1000
    if result is None: return {}, lat
    return {'tool':result.get('tool',''),'arguments':result.get('arguments',{}),'ui_guide':result.get('ui_guide')}, lat

print('Running Qwen3-0.6B BASE on 100 test rows...')
base_results = evaluate('Qwen3-0.6B BASE', predict_base, test_rows)
print('Overall:', base_results['overall'])

[FC_MODEL] Loading model: '/data1/cs24mtech14020/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca'
[FC_MODEL] ENABLE_FC_MODEL='(not set)'
[FC_MODEL] torch 2.6.0+cu118 | CUDA available: True
[FC_MODEL]   GPU 0: NVIDIA RTX A6000 | 47.5 GB VRAM
[FC_MODEL] local_files_only=True | loading tokenizer...
[FC_MODEL] Tokenizer loaded: vocab_size=151643
[FC_MODEL] Loading model onto device='cuda' dtype=torch.float16 ...
[FC_MODEL] Model ready: 596M params on cuda | load took 7.2s
[FC_MODEL] VRAM after load: allocated=1.11 GB | reserved=1.13 GB
[FC_MODEL] --- Inference START ---
[FC_MODEL]   language : 'en'
[FC_MODEL]   query    : 'Looking for interactive learning robot'
[FC_MODEL]   context  : {"currentPage": "product", "selectedCategory": "Humanoid", "lastSearch": "classroom robot kit", "viewedProducts": [10, 1
[FC_MODEL]   thinking mode: DISABLED (enable_thinking=False)
[FC_MODEL]   input tokens : 1317 | max_new_tokens: 300
The following generat

BASE model loaded: True
Running Qwen3-0.6B BASE on 100 test rows...


[FC_MODEL]   output tokens: 46 | latency: 1860ms
[FC_MODEL]   raw output   : '{"tool": "recommend", "arguments": {"need": "interactive learning robot", "budget": 5000, "category": "Humanoid"}, "ui_guide": "open_support/new_ticket/view_tickets"}'
[FC_MODEL]   VRAM used    : 1.12 GB
[FC_MODEL]   result       : tool='recommend' args={'need': 'interactive learning robot', 'budget': 5000, 'category': 'Humanoid'} ui_guide=None
[FC_MODEL] --- Inference END ---
[FC_MODEL] --- Inference START ---
[FC_MODEL]   language : 'en'
[FC_MODEL]   query    : 'Aiper Surfer S1 ke details do'
[FC_MODEL]   context  : {"currentPage": "home", "selectedCategory": "Drone", "lastSearch": "home patrol drone", "viewedProducts": [9, 8], "cart"
[FC_MODEL]   thinking mode: DISABLED (enable_thinking=False)
[FC_MODEL]   input tokens : 1317 | max_new_tokens: 300
[FC_MODEL]   output tokens: 27 | latency: 924ms
[FC_MODEL]   raw output   : '{"tool": "get_product", "arguments": {"product_id": 9}, "ui_guide": "find_drone"}'
[

Overall: {'tool_acc': 0.61, 'arg_f1': 0.4508, 'ui_guide_acc': 0.26, 'p50_ms': 1210.1, 'count': 100}


In [7]:
# ── System 3: Qwen3-0.6B Fine-tuned (ours) ──────────────────────────────────
# QLoRA fine-tuned 3 epochs on 1113 examples. Load base fp16 + apply LoRA adapter.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Free base model from VRAM first
if fc_model._model is not None:
    del fc_model._model; fc_model._model = None
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('Base model freed from VRAM')

BASE_MODEL_PATH = '/data1/cs24mtech14020/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca'
print(f'Loading base model + LoRA adapter from: {ADAPTER_DIR}')

ft_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, local_files_only=True)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token

base_for_ft = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH, torch_dtype=torch.float16, device_map='cuda:0', local_files_only=True
)
ft_model = PeftModel.from_pretrained(base_for_ft, str(ADAPTER_DIR))
ft_model.eval()
print(f'Fine-tuned model ready on {next(ft_model.parameters()).device}')
if torch.cuda.is_available():
    print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')


Base model freed from VRAM
Loading base model + LoRA adapter from: /data1/cs24mtech14020/home/TDL/NexusBots-TDL-Project/finetune/output/qwen3-0_6b-fc-v1


`torch_dtype` is deprecated! Use `dtype` instead!


Fine-tuned model ready on cuda:0
VRAM used: 1.24 GB


In [8]:
def predict_finetuned(query, ctx_str):
    messages = [
        {'role':'system','content':SYSTEM_PROMPT},
        {'role':'user','content':f'Query: {query}\nContext: {ctx_str}'},
    ]
    text = ft_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = ft_tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LENGTH)
    inputs = {k: v.to(ft_model.device) for k,v in inputs.items()}
    start = time.perf_counter()
    with torch.no_grad():
        out = ft_model.generate(**inputs, max_new_tokens=128, do_sample=False, pad_token_id=ft_tokenizer.eos_token_id)
    lat = (time.perf_counter()-start)*1000
    gen = ft_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    r = parse_json(gen)
    return {'tool':r.get('tool',''),'arguments':r.get('arguments',{}),'ui_guide':r.get('ui_guide')}, lat

print('Running Qwen3-0.6B FINE-TUNED on 100 test rows...')
ft_results = evaluate('Qwen3-0.6B FC (ours)', predict_finetuned, test_rows)
print('Overall:', ft_results['overall'])

Running Qwen3-0.6B FINE-TUNED on 100 test rows...
Overall: {'tool_acc': 0.0, 'arg_f1': 0.0, 'ui_guide_acc': 0.27, 'p50_ms': 7298.4, 'count': 100}


In [9]:
# ── Results comparison ───────────────────────────────────────────────────────
all_results = {
    'Heuristic Router': heuristic_results,
    'Qwen3-0.6B BASE': base_results,
    'Qwen3-0.6B FC (ours)': ft_results,
}

print('='*72)
print('B1 — Function-Calling Accuracy  (n=100 holdout test rows)')
print('='*72)
print(f'{"System":<25} {"Tool Acc":>10} {"Arg F1":>10} {"UI Guide":>10} {"p50 ms":>10}')
print('-'*72)
for name, res in all_results.items():
    o = res['overall']
    print(f'{name:<25} {o["tool_acc"]:>10.4f} {o["arg_f1"]:>10.4f} {o["ui_guide_acc"]:>10.4f} {o["p50_ms"]:>10.1f}')

b = base_results['overall']; f = ft_results['overall']
print()
print('='*72)
print('Delta: Fine-tuned vs BASE')
print('='*72)
print(f'  Tool Acc  : {b["tool_acc"]:.4f}  →  {f["tool_acc"]:.4f}   ({f["tool_acc"]-b["tool_acc"]:+.4f})')
print(f'  Arg F1    : {b["arg_f1"]:.4f}  →  {f["arg_f1"]:.4f}   ({f["arg_f1"]-b["arg_f1"]:+.4f})')
print(f'  UI Guide  : {b["ui_guide_acc"]:.4f}  →  {f["ui_guide_acc"]:.4f}   ({f["ui_guide_acc"]-b["ui_guide_acc"]:+.4f})')
print(f'  Latency   : {b["p50_ms"]:.1f} ms  →  {f["p50_ms"]:.1f} ms')

B1 — Function-Calling Accuracy  (n=100 holdout test rows)
System                      Tool Acc     Arg F1   UI Guide     p50 ms
------------------------------------------------------------------------
Heuristic Router              0.4500     0.2522     0.2700        0.0
Qwen3-0.6B BASE               0.6100     0.4508     0.2600     1210.1
Qwen3-0.6B FC (ours)          0.0000     0.0000     0.2700     7298.4

Delta: Fine-tuned vs BASE
  Tool Acc  : 0.6100  →  0.0000   (-0.6100)
  Arg F1    : 0.4508  →  0.0000   (-0.4508)
  UI Guide  : 0.2600  →  0.2700   (+0.0100)
  Latency   : 1210.1 ms  →  7298.4 ms


In [10]:
# ── B4: Multilingual breakdown ───────────────────────────────────────────────
print('B4 — Multilingual Breakdown')
print('='*72)
print(f'{"System":<25} {"Lang":>6} {"Tool Acc":>10} {"Arg F1":>10} {"Count":>8}')
print('-'*72)
for name, res in all_results.items():
    for lang in ['en','hi','te']:
        if lang in res:
            s = res[lang]
            print(f'{name:<25} {lang.upper():>6} {s["tool_acc"]:>10.4f} {s["arg_f1"]:>10.4f} {s["count"]:>8}')
    print()

B4 — Multilingual Breakdown
System                      Lang   Tool Acc     Arg F1    Count
------------------------------------------------------------------------
Heuristic Router              EN     0.4727     0.2518       55
Heuristic Router              HI     0.3333     0.2142       27
Heuristic Router              TE     0.5556     0.3102       18

Qwen3-0.6B BASE               EN     0.6182     0.4606       55
Qwen3-0.6B BASE               HI     0.6296     0.4907       27
Qwen3-0.6B BASE               TE     0.5556     0.3611       18

Qwen3-0.6B FC (ours)          EN     0.0000     0.0000       55
Qwen3-0.6B FC (ours)          HI     0.0000     0.0000       27
Qwen3-0.6B FC (ours)          TE     0.0000     0.0000       18



In [11]:
# ── Save results to CSV + Markdown ───────────────────────────────────────────
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

b1_csv = RESULTS_DIR / 'b1_function_calling.csv'
with open(b1_csv, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['System','Tool Acc','Arg F1','UI Guide Acc','p50 Latency (ms)','Count'])
    for name, res in all_results.items():
        o = res['overall']; w.writerow([name,o['tool_acc'],o['arg_f1'],o['ui_guide_acc'],o['p50_ms'],o['count']])

b1_md = RESULTS_DIR / 'b1_function_calling.md'
with open(b1_md,'w') as f:
    f.write('# B1 — Function-Calling Accuracy\n\n')
    f.write('| System | Tool Acc | Arg F1 | UI Guide Acc | p50 Latency (ms) |\n')
    f.write('|--------|----------|--------|--------------|------------------|\n')
    for name, res in all_results.items():
        o = res['overall']; f.write(f"| {name} | {o['tool_acc']} | {o['arg_f1']} | {o['ui_guide_acc']} | {o['p50_ms']} |\n")

b4_csv = RESULTS_DIR / 'b4_multilingual.csv'
with open(b4_csv,'w',newline='') as f:
    w = csv.writer(f)
    w.writerow(['System','Language','Tool Acc','Arg F1','Count'])
    for name, res in all_results.items():
        for lang in ['en','hi','te']:
            if lang in res:
                s = res[lang]; w.writerow([name,lang.upper(),s['tool_acc'],s['arg_f1'],s['count']])

b4_md = RESULTS_DIR / 'b4_multilingual.md'
with open(b4_md,'w') as f:
    f.write('# B4 — Multilingual Function-Calling Accuracy\n\n')
    f.write('| System | Language | Tool Acc | Arg F1 | Count |\n')
    f.write('|--------|----------|----------|--------|-------|\n')
    for name, res in all_results.items():
        for lang in ['en','hi','te']:
            if lang in res:
                s = res[lang]; f.write(f"| {name} | {lang.upper()} | {s['tool_acc']} | {s['arg_f1']} | {s['count']} |\n")

with open(RESULTS_DIR / 'b1_b4_raw.json','w') as f:
    json.dump(all_results, f, indent=2)

print(f'Saved to {RESULTS_DIR}/')
print(f'  {b1_csv.name}')
print(f'  {b1_md.name}')
print(f'  {b4_csv.name}')
print(f'  {b4_md.name}')
print(f'  b1_b4_raw.json')

Saved to /data1/cs24mtech14020/home/TDL/NexusBots-TDL-Project/research/results/
  b1_function_calling.csv
  b1_function_calling.md
  b4_multilingual.csv
  b4_multilingual.md
  b1_b4_raw.json
